# 01 — Simulated-truth accuracy & calibration (CLAUDE.md §9)

Simulate admixture with known local ancestry (census truth), fit hard-clamp EM on
the references, paint the admixed queries, and score. Strong-structure sims on the
true ARG (the easy regime); inferred-ARG and weak-structure regimes are where tree
accuracy becomes the binding constraint (the §9 head-to-head).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tslai.experiments import admixture_experiment

r = admixture_experiment(T_admix=300, n_admix=6, n_ref=10, sequence_length=4e5,
                         f_A=0.5, max_iter=8, seed=1)
print('queries:', r['n_queries'], 'per-base accuracy:', round(r['accuracy'], 4))
print('Q:', np.round(r['Q'], 6).tolist(), 'pi:', np.round(r['pi'], 3).tolist())

## Calibration (reliability diagram)

In [ ]:
rel = r['reliability']
plt.plot([0, 1], [0, 1], 'k--', label='perfect')
plt.scatter(rel['pred'], rel['emp'], s=20 + 200 * rel['weight'] / rel['weight'].max())
plt.xlabel('predicted P(A)'); plt.ylabel('empirical fraction A'); plt.legend()
plt.title('calibration of the ancestry posterior'); plt.show()

## Headline hypothesis: accuracy vs. admixture age (§9)

In [ ]:
ages = [30, 100, 300, 1000]
accs = []
for T in ages:
    rr = admixture_experiment(T_admix=T, n_admix=6, n_ref=10, sequence_length=4e5,
                              f_A=0.5, max_iter=8, seed=1)
    accs.append(rr['accuracy'])
    print(f'T_admix={T:5d}  accuracy={rr["accuracy"]:.3f}')
plt.plot(ages, accs, 'o-'); plt.xlabel('admixture age (gen)'); plt.ylabel('accuracy')
plt.title('accuracy vs. admixture age (true ARG)'); plt.show()

## A painted haplotype vs. truth

In [ ]:
q = list(r['tracks'].keys())[0]
fig, ax = plt.subplots(figsize=(10, 2))
for seg in r['tracks'][q]:
    ax.plot([seg.left, seg.right], [seg.posterior[0]] * 2, 'C0')
for (l, rr2, st) in r['truth_states'][q]:
    ax.plot([l, rr2], [1 - st] * 2, 'C1', lw=3, alpha=0.4)
ax.set_xlabel('position'); ax.set_ylabel('P(A)'); ax.set_title(f'sample {q}: posterior (blue) vs truth (orange)')
plt.show()